In [1]:
%load_ext cudf.pandas

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [3]:

# %load_ext cudf.pandas
import sys, os
tpch_parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, tpch_parent)
# from tpch import load_lineitem, load_part, q17
import pandas as pd
DATA_ROOT = "/home/jupyter/tpch-dbgen/data"
STORAGE_OPTS = {}



In [4]:

### cell 0 ###

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)


In [5]:

### cell 1 ###

def load_part(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/part.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
part = load_part(DATA_ROOT, **STORAGE_OPTS)

In [6]:
# %%time
# ### cell 2 ###

# # 1. Filter PART to get the relevant keys on GPU
# target_keys = part[(part.P_BRAND == 'Brand#23') & (part.P_CONTAINER == 'MED BOX')].P_PARTKEY

# # 2. Early‐filter LINEITEM to just those keys
# li = lineitem[lineitem.L_PARTKEY.isin(target_keys)]

# # 3. Compute the 20%‐scaled average quantity per key using transform (avoids a merge)
# li['avg'] = li.groupby('L_PARTKEY').L_QUANTITY.transform('mean') * 0.2

# # 4. Apply the quantity filter and compute the final metric entirely on GPU
# good = li[li.L_QUANTITY < li.avg]
# total = pd.DataFrame({'avg_yearly': [good.L_EXTENDEDPRICE.sum() / 7.0]})

In [7]:
%%time
#original
left = lineitem.loc[:, ["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE"]]
right = part[((part["P_BRAND"] == "Brand#23") & (part["P_CONTAINER"] == "MED BOX"))]
right = right.loc[:, ["P_PARTKEY"]]
line_part_merge = left.merge(
    right, left_on="L_PARTKEY", right_on="P_PARTKEY", how="inner"
)
line_part_merge = line_part_merge.loc[
    :, ["L_QUANTITY", "L_EXTENDEDPRICE", "P_PARTKEY"]
]
lineitem_filtered = lineitem.loc[:, ["L_PARTKEY", "L_QUANTITY"]]
lineitem_avg = lineitem_filtered.groupby(
    ["L_PARTKEY"], as_index=False, sort=False
).agg(avg=pd.NamedAgg(column="L_QUANTITY", aggfunc="mean"))
lineitem_avg["avg"] = 0.2 * lineitem_avg["avg"]
lineitem_avg = lineitem_avg.loc[:, ["L_PARTKEY", "avg"]]
total = line_part_merge.merge(
    lineitem_avg, left_on="P_PARTKEY", right_on="L_PARTKEY", how="inner"
)
total = total[total["L_QUANTITY"] < total["avg"]]
total = pd.DataFrame({"avg_yearly": [total["L_EXTENDEDPRICE"].sum() / 7.0]})



CPU times: user 5.18 s, sys: 1.48 s, total: 6.67 s
Wall time: 6.43 s


In [8]:
total

,avg_yearly
0,3.295494e+06


In [7]:
total

,avg_yearly
0,3.295494e+06


In [8]:
### cell 3 ###

total.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   avg_yearly  1 non-null      float64
dtypes: float64(1)
memory usage: 8.0 bytes
